In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import time
import sys
import os
from pixell import enmap, enplot, reproject, utils, curvedsky 
from matplotlib import cm, colormaps
from scipy.optimize import curve_fit
from scipy.linalg import sqrtm

pyilc_folder ='../'

sys.path.insert(1,pyilc_folder+'pyilc/')

from input import ILCInfo
from wavelets import Wavelets, wavelet_ILC, harmonic_ILC

In [ ]:
def download_prepropressed_planckmaps(datafolder='../data/',include_splits = False):
    #in total these maps take up ~2.2 Gb per split (so  ~6.6 Gb in total)
    !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/full/30_full_1024.fits'
    !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/full/44_full_1024.fits'
    !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/full/70_full_1024.fits'
    !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/full/100_full_2048.fits'
    !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/full/143_full_2048.fits'
    !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/full/217_full_2048.fits'
    !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/full/353_full_2048.fits'
    !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/full/545_full_2048.fits'
    !mv *.fits $datafolder
    if include_splits:
        !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/ringhalf1/30_RH1_1024.fits'
        !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/ringhalf1/44_RH1_1024.fits'
        !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/ringhalf1/70_RH1_1024.fits'
        !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/ringhalf1/100_RH1_2048.fits'
        !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/ringhalf1/143_RH1_2048.fits'
        !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/ringhalf1/217_RH1_2048.fits'
        !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/ringhalf1/353_RH1_2048.fits'
        !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/ringhalf1/545_RH1_2048.fits'
        !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/ringhalf2/30_RH2_1024.fits'
        !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/ringhalf2/44_RH2_1024.fits'
        !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/ringhalf2/70_RH2_1024.fits'
        !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/ringhalf2/100_RH2_2048.fits'
        !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/ringhalf2/143_RH2_2048.fits'
        !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/ringhalf2/217_RH2_2048.fits'
        !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/ringhalf2/353_RH2_2048.fits'
        !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/inpainted_input_maps/ringhalf2/545_RH2_2048.fits'
        !mv *.fits $datafolder 

        
        
    return

In [ ]:
download_prepropressed_planckmaps(datafolder,include_splits=False)

In [ ]:
inputfile = pyilc_folder +'input/sample_run_Planck_CMB_HILC.yml'

info_full = ILCInfo(inputfile)

set_freqmapfiles_in_info(info_full,datafolder)

info_full.output_dir = output_folder

info_full.output_prefix = 'full_ILC'

In [ ]:
def run_full_ILC(info):
    ##########################
    # read in frequency maps
    print("reading in maps and other info...")
    info.read_maps()
    # read in bandpasses
    info.read_bandpasses()
    # read in beams
    info.read_beams()
    #########################
    # construct wavelets
    wv = Wavelets(N_scales=info.N_scales, ELLMAX=info.ELLMAX, tol=1.e-6, taper_width=info.taper_width)
    if info.wavelet_type == 'GaussianNeedlets':
        ell, filts = wv.GaussianNeedlets(FWHM_arcmin=info.GN_FWHM_arcmin)
    elif info.wavelet_type == 'CosineNeedlets': # Fiona added CosineNeedlets
        ell,filts = wv.CosineNeedlets(ellmin = info.ellmin,ellpeaks = info.ellpeaks)
    elif info.wavelet_type == 'TopHatHarmonic':
        ell,filts = wv.TopHatHarmonic(info.ellbins)
    else:
        raise TypeError('unsupported wavelet type')
    # wavelet ILC
    if info.wavelet_type == 'TopHatHarmonic':
        print("converting maps to alms...")
        info.maps2alms()
        print("finding C_ells of maps...")
        info.alms2cls()
        print("doing harmonic ILC...")
        harmonic_ILC(wv, info, resp_tol=info.resp_tol, map_images=False)
    else:
        wavelet_ILC(wv, info, resp_tol=info.resp_tol, map_images=False)
    print("done")
    return

In [ ]:
run_full_ILC(info_full)

In [ ]:
CMBmap_full = hp.fitsfunc.read_map(info_full.output_dir+info_full.output_prefix+'needletILCmap_component_CMB.fits')
CMBmap_full_planck = reproject.healpix2map(CMBmap_full, imap.shape, imap.wcs, lmax=6000,rot='gal,equ') ###!!!FIX imap


In [ ]:
def download_preproprecessing_mask(datafolder='../data/'):
    !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/masks/LFI_inpainting_bool.fits'
    !wget 'https://users.flatironinstitute.org/~fmccarthy/ymaps_PR4_McCH23/masks/HFI_inpainting_bool.fits'
    !mv *.fits $datafolder 
    return 

download_preproprecessing_mask(datafolder)
LFImask = hp.fitsfunc.read_map(datafolder+'LFI_inpainting_bool.fits')
HFImask = hp.fitsfunc.read_map(datafolder+'HFI_inpainting_bool.fits')

In [ ]:
%matplotlib inline

In [ ]:
# Defines plotting function for CAR
def eshow(x,**kwargs): 
    ''' Define a function to help us plot the maps neatly '''
    plots = enplot.get_plots(x, **kwargs)
    enplot.show(plots, method = "ipython")

# '''make into method?'''
# # Set the size of the box in degrees and convert to radians
# dec_from, dec_to = np.deg2rad([-40,-10]) ## Inputs declination ()
# ra_from, ra_to = np.deg2rad([-25, 25]) ## Input ascension ()
# box = [[dec_from,ra_from],[dec_to,ra_to]]

# #----------------------------------#
# imap = enmap.read_map(path + fname_dg, box = box)
# # Convert an ndmap map to HEALPix
# smap_healpix = reproject.map2healpix(imap, nside = 512, lmax = 6000)

# # Plot using healpy
# lonra = np.sort(imap.box()[:, 1])/utils.degree
# latra = np.sort(imap.box()[:, 0])/utils.degree
# rang = 300

# hp.cartview(smap_healpix, lonra = lonra, latra = latra, min = -rang, max = rang, 
#             cmap = colormaps.get_cmap('RdYlBu'))

# hp.mollview(smap_healpix, min = -rang, max = rang, 
#             cmap = colormaps.get_cmap('RdYlBu'))

In [ ]:
preprocessing_mask = HFImask*LFImask #hp
hp.mollview(preprocessing_mask,title='mask')
plt.show()

#car
preprocessing_mask_planck = reproject.healpix2map(preprocessing_mask, imap.shape, imap.wcs, lmax=6000,rot='gal,equ') ###!!!FIX imap
eshow(preprocessing_mask_planck, **{"downgrade": 2, "colorbar":True, "ticks": 10, })

In [ ]:
hp.mollview(preprocessing_mask*CMBmap_full,title='HILC CMB') #hp
plt.show()

#car
eshow(preprocessing_mask_planck*CMBmap_full_planck, **{"downgrade": 2, "colorbar":True, "ticks": 10, })

What is the input? HEALPix or CAR? because it looks like pyilc's input.py can take both. 
If the given input is HEALPix, conversation can either be done before run_ilc_full or after.
(maybe we can analyze run time?)

For conversion HEALPix to CAR, figure out the shape&wcs parameters since at the moment they're reliant on nonexistent "imap." Also, figure out if I need "lmax" and "rot" or if they're inherited(?).

Figure out what the splits mean and if they are necessary.

Are we reading in Temperature, Polarization, or Lensing maps? (T, Q, U)